## Feature Engineering

Note: `FeatureEngineer` is the bridge between the cleaned human-readable incident reports and the numerical representation that a machine-learning model can actually consume. If you ask yourself; **Can Logistic Regression train on**?

```python
["pressure",
"leak",
"pipeline"]
```

Answer is `No`. It need numbers.

The most important idea to take note is that:

> **ML models do not understand sentences directly. Vectorization converts text into numerical features while preserving useful information about the words.**

The two methods inside the `FeatureEngineer` class i.e
```python
fit_transform()
transform()
```

are particularly important because they enforce a crucial ML engineering rule:

> **Learn the vocabulary/statistics from the training data, then use that same learned representation to transform validation/test/new data.**

**What is a Vocabulary?**

Take these three reports:

```text
gas leak detected

oil leak detected

gas pressure high
```

The unique words might become:

```text
gas
leak
detected
oil
pressure
high
```

This is our **vocabulary**. We can represent it as:

| Feature  | Index |
| -------- | ----: |
| gas      |     0 |
| leak     |     1 |
| detected |     2 |
| oil      |     3 |
| pressure |     4 |
| high     |     5 |

Now every document can be represented using these features.

---

**What is Bag of Words?**

`Bag of Words` is the simplest approach. The idea is:

> **Count how many times each word occurs in a document.**

Suppose:

```text
Document 1:
gas leak detected

Document 2:
oil leak detected

Document 3:
gas pressure high
```

Vocabulary:

```text
gas
leak
detected
oil
pressure
high
```

The vectors become:

| Document | gas | leak | detected | oil | pressure | high |
| -------- | --: | ---: | -------: | --: | -------: | ---: |
| Doc 1    |   1 |    1 |        1 |   0 |        0 |    0 |
| Doc 2    |   0 |    1 |        1 |   1 |        0 |    0 |
| Doc 3    |   1 |    0 |        0 |   0 |        1 |    1 |

So:

```text
"gas leak detected"
```

becomes:

```text
[1, 1, 1, 0, 0, 0]
```


**Now, suppose our dataset contains:**

```text
report_text
------------------------------------------------
gas leak detected near pipeline
oil leak detected near tank
pipeline pressure is high
```

A machine-learning algorithm cannot directly process:

```text
"gas leak detected near pipeline"
```

Instead, we need something like:

```text
[0.00, 0.71, 0.00, 0.42, ...]
```

which is known as **vectorization**.

The workflow is:

```text
Raw Text
   ↓
Cleaning
   ↓
Tokenization
   ↓
Vocabulary
   ↓
Vectorization
   ↓
Numerical Matrix
   ↓
Machine Learning Model
```


This simple example and explanation should give you a beginner understanding of how vectorization works when working with texts in NLP.

---


**Implementing Bag of Words**

Let's do it manually first with the example below. Create a new `code` cell.

```python
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "gas leak detected",
    "oil leak detected",
    "gas pressure high"
]

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(documents)
```

Now inspect the vocabulary:

```python
print(vectorizer.get_feature_names_out())
```

You might get:

```text
['detected' 'gas' 'high' 'leak' 'oil' 'pressure']
```

Notice that `CountVectorizer` determines the vocabulary automatically.

---

In [1]:
'''Imports setup'''
#!/usr/bin/env python3
import sys
import logging
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)

# Automatically finds the project root 'sira' and adds it to Python's path
PROJECT_ROOT = r"c:\Users\M.faisal\Desktop\sira"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [2]:
documents = [
    "gas leak detected",
    "oil leak detected",
    "gas pressure high"
]

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(documents)

# Now inspect the vocabulary
print(vectorizer.get_feature_names_out())

['detected' 'gas' 'high' 'leak' 'oil' 'pressure']


**Let's look at the Numerical representation**

In [3]:
print(X.toarray())

[[1 1 0 1 0 0]
 [1 0 0 1 1 0]
 [0 1 1 0 0 1]]


*Each row is a document. Each column is a word/feature.*

*What does `fit_transform()` actually do?*

This is one of the most important things to note. When you write:

```python
X_train = vectorizer.fit_transform(train_text)
```

two things happen.

**step 1: `fit()`**

The vectorizer examines the training documents. It learns:

```text
Vocabulary
```

For `TF-IDF` it also learns the statistics required for calculating IDF.

**step 2: `transform()`**

It converts the training documents into vectors using what it just learned. So:

```python
fit_transform()
```

is essentially:

```python
fit()
+
transform()
```

---

**Why NOT use `fit_transform()` everywhere?**

This is a **very important ML engineering concept**. Suppose:

```python
X_train = vectorizer.fit_transform(train_text)
```

This is `correct`.

Then:

```python
X_test = vectorizer.transform(test_text)
```

Also `correct`.

But don't do:

```python
X_test = vectorizer.fit_transform(test_text)
```

`Why`?

Because you are allowing your test dataset to create its own vocabulary. This means your representation of the test data is different from the representation learned from the training data. More importantly, fitting preprocessing on test data can introduce **data leakage**.

The rule is:

```text
TRAINING DATA
      ↓
fit_transform()

VALIDATION DATA
      ↓
transform()

TEST DATA
      ↓
transform()

NEW PRODUCTION DATA
      ↓
transform()
```



**This is one of the concepts you must heavily emphasize as you progress**.

---

## TF-IDF

Now we move to the more sophisticated approach.

**TF-IDF** means:

> **Term Frequency – Inverse Document Frequency**

The fundamental idea is:

> A word is more useful when it is frequent in a particular document but not common across every document.


**Why do we need TF-IDF?**

Consider:

```text
incident detected

gas leak detected

oil leak detected
```

The word:

```text
detected
```

appears everywhere.

It doesn't tell us much about the difference between the documents.

But:

```text
gas
oil
leak
```

may be more informative. `TF-IDF` reduces the importance of words that occur across many documents.

**Term Frequency**

TF asks:

> How frequently does this word appear in this document?

For example:

```text
gas gas gas leak
```

The word:

```text
gas
```

has a high term frequency.

**Inverse Document Frequency**

IDF asks approximately:

> How rare is this word across the collection of documents?

If a word occurs in almost every document:

```text
incident
```

its IDF becomes relatively low. If a word appears in very few documents:

```text
corrosion
```

its IDF becomes higher. Therefore:

```text
TF × IDF
```

gives us the TF-IDF score.

---

**Implement TF-IDF**

add this import: `from sklearn.feature_extraction.text import TfidfVectorizer`

In [4]:
documents = [
    "gas leak detected",
    "oil leak detected",
    "gas pressure high"
]

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(documents)

In [5]:
# View vocabulary
print(vectorizer.get_feature_names_out())

['detected' 'gas' 'high' 'leak' 'oil' 'pressure']


In [6]:
# View vectors
print(X.toarray())

[[0.57735027 0.57735027 0.         0.57735027 0.         0.        ]
 [0.51785612 0.         0.         0.51785612 0.68091856 0.        ]
 [0.         0.4736296  0.62276601 0.         0.         0.62276601]]


The values will now be decimal numbers rather than simple counts. For example:

```text
[
    [0.577, 0.577, 0.000, 0.577, 0.000, 0.000],
    ...
]
```

The exact numbers depend on the corpus and vectorizer settings.

---

**Bag of Words vs TF-IDF**

| Property            | Bag of Words     | TF-IDF                     |
| ------------------- | ---------------- | -------------------------- |
| Representation      | Word counts      | Weighted word importance   |
| Common words        | High counts      | Down-weighted              |
| Rare words          | Counts frequency | Often receives more weight |
| Simplicity          | Very simple      | More sophisticated         |
| Interpretability    | High             | High                       |
| Text classification | Good baseline    | Often stronger             |
| Computation         | Fast             | Slightly more complex      |


**Applying This to SIRA**

Our actual workflow becomes:

```text
incident_reports_clean.csv
          ↓
      DataLoader
          ↓
    Preprocessing
          ↓
    FeatureEngineer
          ↓
      TF-IDF
          ↓
   Numerical Features
          ↓
     ML Classifier
          ↓
   Incident Prediction
```

---

### Create/update the `feature_engineering.py` module with the script below.

```python
"""
feature_engineering.py

Feature engineering module for Sira.

This module converts text into numerical features using:
1. TF-IDF
2. Bag of Words

The same fitted vectorizer must be used when transforming
training, validation, testing, and new production data.
"""

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    CountVectorizer
)


class FeatureEngineer:
    """
    Convert incident report text into numerical features.
    """

    def __init__(
        self,
        method="tfidf",
        max_features=None,
        ngram_range=(1, 1)
    ):
        """
        Initialize the feature engineering pipeline.

        Parameters
        ----------
        method : str
            Vectorization method.
            Options:
                - "tfidf"
                - "bow"

        max_features : int or None
            Maximum number of vocabulary features.

        ngram_range : tuple
            Range of n-grams to generate.
            (1, 1) = unigrams
            (1, 2) = unigrams + bigrams
        """

        self.method = method.lower()

        self.max_features = max_features

        self.ngram_range = ngram_range

        # Create the appropriate vectorizer
        if self.method == "tfidf":

            self.vectorizer = TfidfVectorizer(
                max_features=self.max_features,
                ngram_range=self.ngram_range
            )

        elif self.method == "bow":

            self.vectorizer = CountVectorizer(
                max_features=self.max_features,
                ngram_range=self.ngram_range
            )

        else:

            raise ValueError(
                "method must be either 'tfidf' or 'bow'"
            )

    # FIT + TRANSFORM
    def fit_transform(self, texts):
        """
        Learn the vocabulary from the supplied text
        and transform the text into numerical features.

        This should normally be used on TRAINING data.
        """

        return self.vectorizer.fit_transform(texts)

    # TRANSFORM
    def transform(self, texts):
        """
        Transform text using the vocabulary/statistics
        learned during fit_transform().

        This should be used on validation, test,
        and new/unseen data.
        """

        return self.vectorizer.transform(texts)

    # FEATURE NAMES
    def get_feature_names(self):
        """
        Return the vocabulary/features learned
        by the vectorizer.
        """

        return self.vectorizer.get_feature_names_out()

    # VOCABULARY SIZE
    def get_vocabulary_size(self):
        """
        Return the number of features learned.
        """

        return len(
            self.vectorizer.get_feature_names_out()
        )
```

---

**Using your `FeatureEngineer`**

Suppose:

```python
texts = df["report_text"]
```

In [7]:
import os
print(os.getcwd())

c:\Users\M.faisal\Desktop\Applied-ML\smart-incident-report-analyzer


In [8]:
import pandas as pd

df = pd.read_csv(r"c:\Users\M.faisal\Desktop\Applied-ML\smart-incident-report-analyzer\data\processed\incident_reports_clean.csv", index_col=False)

In [9]:
#Load few data heads
df["report_text"].head(10)

0    pressure leak detected on pipeline 7 during ro...
1         smoke observed from electrical control panel
2       worker slipped on wet surface near loading bay
3               oil spill discovered near storage tank
4      corrosion observed on external pipeline coating
5         smoke observed from electrical control panel
6       worker slipped on wet surface near loading bay
7    fire extinguisher inspection completed success...
8     loose flange bolts discovered during maintenance
9     abnormal temperature recorded in processing unit
Name: report_text, dtype: str

In [10]:
#getting the reports_text features
texts = df["report_text"]

Create:

```python
feature_engineer = FeatureEngineer(
    method="tfidf"
)
```

Then:

```python
X = feature_engineer.fit_transform(texts)
```

In [12]:
from src.feature_engineering import FeatureEngineer


feature_engineer = FeatureEngineer(
    method="tfidf"
)


X = feature_engineer.fit_transform(texts)

In [13]:
# Check
print(X.shape)

(924, 75)


You might get a different shape e.g:

```text
(1000, 85)
```

This means:

```text
1000 documents
85 features
```

The exact number depends on your cleaned dataset and vectorizer settings.

---

In [15]:
# Inspect the Features
features = feature_engineer.get_feature_names()

print(features)

['abnormal' 'acceptable' 'activated' 'after' 'alarm' 'area' 'bay'
 'beneath' 'bolts' 'calibration' 'coating' 'completed' 'compressor'
 'control' 'corrosion' 'crane' 'detected' 'detector' 'discovered' 'during'
 'electrical' 'emergency' 'entered' 'exceeded' 'external' 'extinguisher'
 'fire' 'flange' 'fluid' 'found' 'from' 'gas' 'hydraulic' 'in'
 'inspection' 'issues' 'leak' 'loading' 'loose' 'maintenance' 'near'
 'observed' 'oil' 'on' 'panel' 'personnel' 'pipeline' 'pressure'
 'processing' 'pump' 'recorded' 'restricted' 'routine' 'sensor' 'shutdown'
 'slipped' 'small' 'smoke' 'spill' 'station' 'storage' 'successfully'
 'surface' 'surge' 'tank' 'temperature' 'threshold' 'triggered'
 'unauthorized' 'unit' 'valve' 'vibration' 'wet' 'without' 'worker']


You might see:

```text
[
    'abnormal',
    'alarm',
    'corrosion',
    'detected',
    'electrical',
    'gas',
    'leak',
    'pipeline',
    'pressure',
    ...etc
]
```

These are now the **features** that the ML model sees.

---

In [16]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5608 stored elements and shape (924, 75)>
  Coords	Values
  (0, 47)	0.341593840837966
  (0, 36)	0.33456938317960805
  (0, 16)	0.42817523754411274
  (0, 43)	0.3034575710159748
  (0, 46)	0.35201789241814746
  (0, 19)	0.3462143669720921
  (0, 52)	0.35201789241814746
  (0, 34)	0.3581324660298129
  (1, 57)	0.4186369917290572
  (1, 41)	0.351731923174776
  (1, 30)	0.4186369917290572
  (1, 20)	0.4186369917290572
  (1, 13)	0.4186369917290572
  (1, 44)	0.4186369917290572
  (2, 43)	0.2711379369543011
  (2, 74)	0.3775157899201557
  (2, 55)	0.3775157899201557
  (2, 72)	0.3775157899201557
  (2, 62)	0.3775157899201557
  (2, 40)	0.26716135444695266
  (2, 37)	0.3775157899201557
  (2, 6)	0.3775157899201557
  (3, 40)	0.31149599587291954
  (3, 42)	0.44016342551624654
  (3, 58)	0.44016342551624654
  :	:
  (920, 32)	0.38627259470467007
  (920, 28)	0.38627259470467007
  (920, 29)	0.38627259470467007
  (920, 7)	0.38627259470467007
  (920, 15)	0.386

**Important: Sparse Matrices**

You may notice that:

```python
print(X)
```

doesn't display a normal DataFrame. That's because text vectorization usually produces a **sparse matrix**. Imagine:

```text
1000 documents × 10,000 words
```

That's:

```text
10,000,000 values
```

But most documents contain only a small fraction of those words. So instead of storing thousands of zeros, Scikit-learn uses a sparse representation.

This:

```text
[0, 0, 0, 1, 0, 0, 0, 0, 1, 0, ...]
```

is stored efficiently. You can convert a small matrix to dense form for learning:

```python
X.toarray()
```

But **don't routinely convert large text datasets to dense arrays**, because memory usage can explode.

---

In [18]:
# X_densed = X.toarray()
# X_densed

## QUICK NOTE

*The vectorizer does not automatically take all 1,000 columns/fields in the cleaned dataset.* We normally choose which column(s) contain the text that we want to convert into numerical features.

For SIRA, since our dataset looks roughly like:

```text
incident_id
report_date
location
department
severity
status
shift
report_text
```

then **TF-IDF or Bag of Words should primarily operate on `report_text`**, because that is the unstructured `natural-language` information.

**The typical SIRA pipeline**

```text
1000 incident records
        │
        ▼
Clean DataFrame
        │
        ├── report_date ──────► date processing
        ├── location ─────────► categorical processing
        ├── department ───────► categorical processing
        ├── severity ─────────► categorical processing
        ├── status ───────────► categorical processing
        ├── shift ────────────► categorical processing
        │
        └── report_text ──────► TF-IDF / BOW
                                  │
                                  ▼
                           Numerical features
```

So if we do:

```python
X = feature_engineer.fit_transform(
    df["report_text"]
)
```

we're now saying:

> "Take the 1,000 incident reports and learn a vocabulary from their text, then convert each report into a numerical vector."

If there are 1,000 rows and the vectorizer learns 500 useful terms, you might get:

```text
X.shape
```

```text
(1000, 500)
```

That means:

* **1,000 rows** = 1,000 incidents
* **500 columns** = 500 text features learned from `report_text`

---

**But here's the important ML engineering question**

Question we should ask now is: `What about the other columns?`

We **can and often should use them**, but we don't send all of them through TF-IDF.

For example:

| Column        | Appropriate treatment                            |
| ------------- | ------------------------------------------------ |
| `report_text` | TF-IDF / BOW                                     |
| `department`  | One-Hot Encoding                                 |
| `location`    | One-Hot Encoding                                 |
| `severity`    | Encoding / ordinal encoding depending on meaning |
| `status`      | One-Hot Encoding                                 |
| `shift`       | One-Hot Encoding                                 |
| `report_date` | Extract year/month/day/day-of-week/etc.          |
| `incident_id` | Usually exclude                                  |
| Target/label  | Keep separately as `y`                           |

Then you combine the resulting features.

Conceptually:

```text
                 Incident Dataset
                       │
          ┌────────────┼────────────┐
          │            │            │
          ▼            ▼            ▼
     report_text   categorical     dates
          │            │            │
          ▼            ▼            ▼
        TF-IDF       Encoder      Features
          │            │            │
          └────────────┼────────────┘
                       ▼
                Combined Features
                       │
                       ▼
                  ML Algorithm
```

**For example**

Suppose one incident is:

```text
report_text: "High pressure detected near pipeline"

department: "Pipeline Operations"

severity: "High"

shift: "Night"
```

The final ML representation could combine:

```text
TF-IDF features + department features + severity features + shift features + date-derived features
```

and produce one large numerical feature vector for that incident.

---

**This is actually where SIRA becomes more interesting**. Our current `FeatureEngineer` is focused on:

```python
df["report_text"]
```

This is perfectly appropriate for introducing **text vectorization**. But we'll **not stop here** since the goal is to teach you real ML Engineering. Hence, the next evolution of `FeatureEngineer` should eventually handle:

```text
                    FeatureEngineer
                          │
              ┌───────────┴───────────┐
              │                       │
         Text Features          Structured Features
              │                       │
           TF-IDF                 Encoding
           BOW                    Scaling
           N-grams                Date features
              │                       │
              └───────────┬───────────┘
                          ▼
                   Final Feature Matrix
                          │
                          ▼
                    ML Classifier
```

And this is also where **`ColumnTransformer`** from scikit-learn becomes very useful. For SIRA, we'll introduce it **before training the final model**, because it will teach the you how real-world datasets containing both **unstructured text + structured/tabular data** are handled.

One important correction to keep in mind: **the vectorizer can process multiple text columns if we explicitly combine or separately vectorize them, but it will not intelligently decide how to process every column in our DataFrame.** To do that we have to define the preprocessing strategy for each feature type which is beyond our learning now.

---

This project gives you your **first complete NLP pipeline**, built with clean software engineering practices. You'll finish Phase 3 with reusable modules that will plug directly into **Phase 4**, where you'll train and evaluate your first text classification model using the engineered features.

---


**Updated `project structure`**

By the end of phase 3:

```text
smart_incident_report_analyzer/

│
├── data/
│   ├── ***
|
├── models/
│
├── notebooks/
|   ├── ***
│
├── src/
│   ├── __init__.py
│   ├── incident.py
|   ├── clean_data.py
│   ├── data_loader.py
│   ├── eda.py
│   ├── preprocessing.py
│   ├── feature_engineering.py   # New
│   └── utils.py
│
├── main.py
├── README.md
├── requirements.txt
├── .gitignore
└── ***
```